# Data Cleaning and Preprocessing with Python — Practice Notebook

**Course:** Data Analysis with Python (DSAI1005)  
**Lecturer:** Dr. Minh Duc Vu (`minhvd@neu.edu.vn`)  
**Department:** School of Data Science and Artificial Intelligence – National Economics University (NEU)  
**Suggested Duration:** 150–180 minutes  
**Context:** Business analytics, customer data, sales records, and transactions  
**Data Ecosystem:** Python 3, NumPy, Pandas, Matplotlib

This notebook accompanies the lecture on **Data Preprocessing and Data Cleaning in Python**.

Each section is structured sequentially:

1. Theoretical recap and core concept;
2. An executable working example;
3. Hands-on exercises placed immediately after the concept;
4. Guided `TODO` starter code with blanks (`________`);
5. An integrated comprehensive pipeline project at the end.

> **Recommended Learning Workflow:** Run the notebook from top to bottom once to understand the overall pipeline. Then return to each cell marked `TODO`, fill in the blanks, and execute the tests.

## 0. Environment Setup

Import standard libraries for tabular data processing and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

# 1. Create a Realistic Dirty Dataset

To practice data cleaning effectively, we initialize a synthetic e-commerce transaction dataset with common real-world flaws:

- Inconsistent column names and casing;
- Duplicate records;
- Inconsistent categorical strings (`"Hanoi"`, `"HANOI"`, `"ha nei"`, `"Cash"`, `"cash"`, `"CASH"`);
- Mixed date representations (`"2026-09-01"`, `"02/09/2026"`, `"Sep 3, 2026"`, `"invalid_date"`);
- Missing values in multiple columns;
- Invalid values (`Quantity = -1`, negative revenue);
- Mixed measurement units (`kg` vs. `g`);
- Business rule violations ($Revenue \neq Quantity \times Unit\ Price$).

In [ ]:
raw_data = {
    "Order ID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015",
        "O005"  # duplicate row
    ],
    "Customer ID": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010",
        "C011", "C012", "C013", "C014", "C015",
        "C005"
    ],
    "Region": [
        "Hanoi", "HANOI", "ha noi", "Ho Chi Minh", "HO CHI MINH",
        "ho chi minh", "Da Nang", "da nang", "Hanoi", "Hanoi",
        np.nan, "Da Nang", "HANOI", "Ho Chi Minh", "Da Nang",
        "HO CHI MINH"
    ],
    "Order Date": [
        "2026-09-01", "02/09/2026", "Sep 3, 2026", "2026-09-04", "05/09/2026",
        "2026-09-06", "07-09-2026", "2026/09/08", "2026-09-09", "2026-09-10",
        "2026-09-11", "2026-09-12", "invalid_date", "2026-09-14", "2026-09-15",
        "05/09/2026"
    ],
    "Product": [
        "Coffee", "Tea", "Coffee", "Juice", "Tea",
        "Coffee", "Juice", "Tea", "Coffee", "Tea",
        "Coffee", "Juice", "Tea", "Coffee", "Tea",
        "Tea"
    ],
    "Quantity": [
        2, 3, 1, 4, 2,
        5, 3, 2, 4, 100,  # bulk order / potential outlier
        2, 3, -1, 1, 2,   # -1 is invalid quantity
        2
    ],
    "Unit Price": [
        50000, 40000, 50000, 60000, np.nan,
        50000, 60000, 40000, 50000, 40000,
        50000, np.nan, 40000, 50000, np.nan,
        np.nan
    ],
    "Revenue": [
        100000, 120000, 50000, 240000, 80000,
        250000, 180000, 80000, -50000, 4000000,  # negative revenue and extreme high
        100000, 180000, 40000, 999999, 80000,     # 999999 mismatch
        80000
    ],
    "Payment Method": [
        "Cash", "cash", "CASH", "Credit Card", "credit card",
        "Bank Transfer", "bank transfer", "Cash", "cash", "Credit Card",
        "Bank Transfer", np.nan, "credit card", "Cash", "credit card",
        "credit card"
    ],
    "Weight": [
        1.2, 800, 1.5, 2.0, 650,
        1.0, 900, 0.8, 1.1, 2.5,
        1.4, 600, 1.0, 1.3, 650,
        650
    ],
    "Weight Unit": [
        "kg", "g", "kg", "kg", "g",
        "kg", "g", "kg", "kg", "g",
        "kg", "kg", "g", "kg", "kg",
        "g"
    ]
}

df = pd.DataFrame(raw_data)
df

# 2. Initial Data Quality Assessment

### Theory Recap

Before modifying any data, always answer the following essential questions:

- How many rows and columns are in the dataset?
- What are the current data types?
- Are there missing values?
- Are there duplicate records?
- Which columns are numerical vs. categorical variables?

Commonly used methods:

```python
df.shape
df.head()
df.info()
df.isna().sum()
df.duplicated().sum()
```

In [ ]:
# Example: Inspect the first 5 rows
df.head()

In [ ]:
# TODO 1:
# 1. Print dataset dimensions (shape)
# 2. Inspect structure using info()
# 3. Count missing values per column
# 4. Count duplicate rows

print("Shape:", ________)

# df.________()

# print("\nMissing values:")
# print(df.________().sum())

# print("\nDuplicate rows:", df.________().sum())

### Questions for Reflection

1. Which columns currently have missing values?
2. What is the current data type of `Order Date`?
3. Can we immediately conclude that any numeric column is a numerical feature? Why or why not?

# 3. Standardizing Column Names

### Theory Recap

Consistent column names make code readable, reduce errors, and allow attribute-style access.

A standard naming convention (snake_case):

```text
"Order ID"       → "order_id"
"Payment Method" → "payment_method"
```

Useful string methods:

```python
.str.strip()
.str.lower()
.str.replace()
```

In [ ]:
# Example
example_columns = pd.Index([" Customer Name ", "Order Value", "Region-Code"])

(
    example_columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

In [ ]:
# TODO 2:
# Standardize all column names of df
# Hint: strip -> lower -> replace space -> replace hyphen

df.columns = (
    df.columns
    .str.________()
    .str.________()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns

# 4. Checking and Handling Duplicate Records

### Theory Recap

`duplicated()` returns `True` for rows considered duplicates.

```python
df.duplicated()
df.duplicated().sum()
```

However, duplicates must always be evaluated in **business domain context**.

In transaction data, two identical rows might still be two legitimate separate transactions if they have different `order_id` values.

Here, `order_id` serves as the primary business entity key.

In [ ]:
# Inspect duplicated Order IDs
df[df.duplicated(subset=["order_id"], keep=False)]

In [ ]:
# TODO 3:
# 1. Count duplicated Order IDs
# 2. Create df_no_duplicates by keeping the first occurrence

duplicate_order_count = df.________(
    subset=["order_id"]
).sum()

print("Duplicate Order IDs:", duplicate_order_count)

df_no_duplicates = df.________(
    subset=["order_id"],
    keep="first"
).copy()

print("Shape before:", df.shape)
print("Shape after :", df_no_duplicates.shape)

# 5. Identifying Numerical and Categorical Variables

### Theory Recap

We can filter columns by data type:

```python
df.select_dtypes(include=np.number)
df.select_dtypes(include=["object", "category"])
```

Important domain caveat:

> `Customer_ID` or `Order_ID` may be stored as numeric or string types, but conceptually they are **identifiers** (keys), not continuous numerical features for which calculating statistics like mean or median makes sense.

In [ ]:
# TODO 4:
# Identify numerical and categorical columns

num_cols = df_no_duplicates.select_dtypes(
    include=________
).columns.tolist()

cat_cols = df_no_duplicates.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

# 6. Checking Unique Values in Categorical Variables

### Theory Recap

Helpful methods:

```python
nunique()
unique()
value_counts()
```

These reveal variant labels such as:

```text
"Hanoi"
"HANOI"
" Hanoi "
"ha noi"
```

which represent the exact same geographical entity.

In [ ]:
# Example: inspect Region values
df_no_duplicates["region"].value_counts(dropna=False)

In [ ]:
# TODO 5:
# Inspect unique values of payment_method

print(
    df_no_duplicates["payment_method"].________()
)

# 7. Standardizing Text Data

### Theory Recap

Standard text operations in Pandas:

```python
.str.strip()
.str.lower()
.str.upper()
.str.title()
.replace()
```

The standard sequence:

```text
strip whitespace
→ standardize casing
→ map equivalent variants
```

In [ ]:
# Example: standardize payment_method
df_clean = df_no_duplicates.copy()

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .str.strip()
    .str.lower()
)

df_clean["payment_method"].value_counts(dropna=False)

In [ ]:
# TODO 6:
# Standardize Region:
# 1. strip()
# 2. lower()
# 3. map "ha noi" -> "hanoi"
# 4. "ho chi minh" stays as is
# 5. "da nang" stays as is

df_clean["region"] = (
    df_clean["region"]
    .str.________()
    .str.________()
)

df_clean["region"] = df_clean["region"].replace({
    "________": "hanoi"
})

df_clean["region"].value_counts(dropna=False)

# 8. Standardizing Date Data

### Theory Recap

Dates are often read from CSV as `object` (string) types.

We convert them with:

```python
pd.to_datetime(..., errors="coerce")
```

`errors="coerce"` converts unparseable strings into `NaT` (Not a Time).

Then datetime properties become accessible:

```python
.dt.year
.dt.month
.dt.day
```

In [ ]:
# Inspect current date values
df_clean["order_date"]

In [ ]:
# TODO 7:
# Convert order_date to datetime
# Hint: format="mixed" is helpful when multiple date formats are present

df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    format="mixed",
    dayfirst=True,
    errors="________"
)

print(df_clean["order_date"])
print("\nInvalid dates:", df_clean["order_date"].isna().sum())

# 9. Detecting Missing Values

### Theory Recap

Two equivalent detection functions:

```python
df.isnull()
df.isna()
```

To compute missing proportions:

```python
df.isna().mean() * 100
```

Note: Missing values can also be disguised as `"Unknown"`, `-1`, `999`, `"N/A"`, etc., which Pandas may not automatically detect as `NaN`.

In [ ]:
# TODO 8:
# Create a summary table of missing counts and percentages by column

missing_count = df_clean.________().sum()
missing_percent = df_clean.________().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent.round(2)
})

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

# 10. Handling Missing Values

### Theory Recap

There is no single universal method for missing values.

- Numerical variables:
  - `mean`;
  - `median`.
- Categorical variables:
  - `mode`;
  - `"unknown"` / separate category.
- Dropping rows or columns when missingness is negligible or the feature is redundant.

Median is often preferable over mean when distributions are skewed or contain outliers.

In [ ]:
# Example: impute payment_method with its mode
payment_mode = df_clean["payment_method"].mode()[0]

df_clean["payment_method"] = (
    df_clean["payment_method"]
    .fillna(payment_mode)
)

payment_mode

In [ ]:
# TODO 9:
# Impute missing Unit Price with the median Unit Price grouped by Product
# Hint: groupby("product")["unit_price"].transform("median")

median_price_by_product = (
    df_clean
    .groupby("product")["unit_price"]
    .transform("________")
)

df_clean["unit_price"] = (
    df_clean["unit_price"]
    .fillna(________)
)

df_clean[["product", "unit_price"]]

In [ ]:
# TODO 10:
# For missing Region, impute with the mode of Region

region_mode = df_clean["region"].________()[0]

df_clean["region"] = (
    df_clean["region"]
    .fillna(________)
)

df_clean["region"].value_counts(dropna=False)

# 11. Checking Valid Ranges

### Theory Recap

A value might not be missing, yet completely invalid in reality.

Examples:

```text
Age < 0
Quantity <= 0
Score > 100
```

Useful checking methods:

```python
between()
df[df["quantity"] <= 0]
```

In [ ]:
# TODO 11:
# Display rows with Quantity <= 0

invalid_qty = df_clean[
    df_clean["quantity"] ________ 0
]

invalid_qty

### Questions for Reflection

What plausible explanations could exist for `Quantity = -1`?

- A data entry typo?
- A customer return or refund?
- An inventory adjustment?

Never automatically delete records before understanding the business rule.

# 12. Validating Business Rules: Revenue

### Theory Recap

Business domain rules help detect corrupted or fabricated records.

In this transaction dataset:

$$
Revenue \approx Quantity \times Unit\ Price
$$

Large discrepancies require investigation.

In [ ]:
# Calculate expected revenue
df_clean["expected_revenue"] = (
    df_clean["quantity"]
    * df_clean["unit_price"]
)

df_clean[
    [
        "order_id",
        "quantity",
        "unit_price",
        "revenue",
        "expected_revenue"
    ]
]

In [ ]:
# TODO 12:
# Create column revenue_diff = revenue - expected_revenue
# Display rows where absolute difference > 1

df_clean["revenue_diff"] = (
    df_clean["________"]
    - df_clean["________"]
)

revenue_mismatch = df_clean[
    df_clean["revenue_diff"].abs() > ________
]

revenue_mismatch

# 13. Outlier Detection with Boxplots

### Theory Recap

Boxplots display:

- Q1 (25th percentile);
- Median (50th percentile);
- Q3 (75th percentile);
- Whiskers;
- Potential outliers beyond whiskers.

Points outside whiskers are **not automatically data errors**.

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(
    df_clean["revenue"].dropna(),
    vert=False
)
plt.xlabel("Revenue")
plt.title("Boxplot of Revenue")
plt.show()

# 14. Outlier Detection with IQR

### Theory Recap

$$
IQR = Q_3 - Q_1
$$

$$
Lower = Q_1 - 1.5 \times IQR
$$

$$
Upper = Q_3 + 1.5 \times IQR
$$

IQR is less sensitive to extreme values than methods based on mean and standard deviation.

In [ ]:
# TODO 13:
# Calculate IQR for Revenue

q1 = df_clean["revenue"].quantile(________)
q3 = df_clean["revenue"].quantile(________)

iqr = ________ - ________

lower = q1 - 1.5 * ________
upper = q3 + 1.5 * ________

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower:", lower)
print("Upper:", upper)

In [ ]:
# TODO 14:
# Display potential outliers

revenue_outliers = df_clean[
    (df_clean["revenue"] < ________)
    |
    (df_clean["revenue"] > ________)
]

revenue_outliers

### Questions for Reflection

1. Should we immediately delete `Revenue = 4,000,000`?
2. Could this be a legitimate large wholesale order?
3. If this dataset is intended for fraud detection, what consequences would deleting outliers bring?

# 15. Standardizing Measurement Units

### Theory Recap

Inconsistent measurement units can produce false outliers.

For instance:

```text
1.2 kg
800 g
1.5 kg
650 g
```

All values must be harmonized into the same unit before analysis.

In [ ]:
# Inspect current weight data
df_clean[["weight", "weight_unit"]]

In [ ]:
# TODO 15:
# Convert all weights to kg.
# If weight_unit == "g", divide weight by 1000.
# Then set all weight_unit values to "kg".

mask_g = df_clean["weight_unit"] == "g"

df_clean.loc[
    mask_g,
    "weight"
] = (
    df_clean.loc[
        mask_g,
        "weight"
    ] / ________
)

df_clean["weight_unit"] = "________"

df_clean[["weight", "weight_unit"]]

# 16. Post-Cleaning Data Validation

### Theory Recap

After cleaning, re-verify data integrity:

```text
missing values?
duplicates?
invalid ranges?
wrong data types?
unexpected categories?
business rule violations?
```

Assertions enable automated validation checks:

```python
assert condition
```

In [ ]:
# Summary validation checks
print("Duplicate Order IDs:",
      df_clean.duplicated(subset=["order_id"]).sum())

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nData types:")
print(df_clean.dtypes)

In [ ]:
# TODO 16:
# Complete appropriate assertions

assert df_clean.duplicated(
    subset=["order_id"]
).sum() == ________

assert (df_clean["unit_price"] > 0).________()

assert df_clean["region"].isna().sum() == ________

print("Basic validation assertions passed successfully.")

# 17. Post-Cleaning Dataset Summary

Review the cleaned dataset:

In [ ]:
df_clean

In [ ]:
# TODO 17:
# Generate descriptive statistics for numerical variables
df_clean.________()

# 18. Comprehensive Practice Exercise

Attempt to complete this exercise without referring back to previous cells if possible.

## Requirements

From the initial `df`, create a new DataFrame named:

```python
final_df
```

and execute the entire data cleaning pipeline:

1. Standardize column names.
2. Deduplicate `order_id` records.
3. Standardize `region`.
4. Standardize `payment_method`.
5. Convert `order_date` to datetime.
6. Impute missing `unit_price`.
7. Impute missing `region`.
8. Impute missing `payment_method`.
9. Check `quantity <= 0`.
10. Convert all weights to kilograms (kg).
11. Calculate `expected_revenue`.
12. Identify rows violating business rules.
13. Calculate IQR of `revenue`.
14. Identify potential outliers.
15. Validate the dataset.

> **Crucial Directive:** Do not automatically remove outliers without a sound business reason.

In [ ]:
# TODO 18 - Comprehensive Practice Exercise
# Write your full end-to-end cleaning pipeline here.

final_df = df.copy()

# 1. Standardize column names
# ...

# 2. Deduplicate order_id
# ...

# 3. Standardize region
# ...

# 4. Standardize payment_method
# ...

# 5. Convert order_date
# ...

# 6-8. Missing values
# ...

# 9. Invalid quantity
# ...

# 10. Weight -> kg
# ...

# 11-12. Business rules
# ...

# 13-14. Outliers
# ...

# 15. Validation
# ...

final_df.head()

# 19. Self-Assessment Conceptual Questions

Answer in Markdown directly beneath each question.

1. How does Data Cleaning differ from Data Preprocessing?
2. Why should `drop_duplicates()` never be called mechanically without domain checks?
3. Why should `Customer_ID` not be treated as a standard numerical variable?
4. Is there any difference between `isna()` and `isnull()` in Pandas?
5. Why is the median often better than the mean when data contains outliers or skewness?
6. Why is an outlier not necessarily an error?
7. Why must measurement units be standardized before outlier detection?
8. What does `errors="coerce"` do in `pd.to_datetime()`?
9. What role do business rules play in Data Validation?
10. Why is post-cleaning validation still necessary after cleaning steps?

# 20. Extension Challenges

If you have completed the core requirements, try these advanced exercises:

### Challenge A — Capping Outliers (Winsorization)

Instead of deleting revenue outliers, use:

```python
Series.clip()
```

to cap revenue within `[lower, upper]`.

### Challenge B — Reusable Text Cleaning Function

Write a reusable helper function:

```python
def clean_text_column(series):
    ...
```

to strip whitespace, lowercase text, and standardize spaces.

### Challenge C — Automated Validation Suite

Write a function:

```python
def validate_sales_data(df):
    ...
```

that returns a summary dictionary like:

```python
{
    "duplicate_order_id": ...,
    "missing_unit_price": ...,
    "invalid_quantity": ...,
    "revenue_mismatch": ...
}
```

# 21. Pre-Submission Checklist

```text
□ I checked dataset shape and info()
□ I checked duplicates using appropriate keys
□ I distinguished numerical vs. categorical variables
□ I checked unique values across categorical features
□ I standardized string text
□ I standardized column names into snake_case
□ I converted dates to datetime objects
□ I checked missing values and calculated percentages
□ I selected appropriate imputation strategies
□ I checked for invalid values and domain ranges
□ I checked and verified business rules
□ I checked potential outliers using IQR and boxplot
□ I refrained from blindly removing outliers
□ I standardized measurement units into a common scale
□ I validated the final dataset with assertions
```